# 01 — Data audit

**Phase 0 · MetaFlex Decoded**

The point of this notebook is not to clean anything. It is to find out what is actually
in these files, so that every cleaning decision later is a decision and not a guess.

By the end I should be able to answer:

1. How many patients, how many patient-visits, how many files?
2. What are the columns, in both layers, and what do they mean?
3. How much data is missing, and where?
4. Are the timestamps regular? Where are the gaps?
5. Which files are `.xls` and which are `.xlsx`, and does it matter?
6. What surprised me?

Anything I decide here gets written into `docs/data-decisions.md`.

## Setup

In [ ]:
import glob
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")     # notebook lives in notebooks/, data lives in data/raw/

print("Looking in:", RAW.resolve())
print("Exists:", RAW.exists())


### If that printed `Exists: False`

The path is relative to where this notebook file sits. If your notebook is in
`metaflex-decoded/notebooks/` and the data is in `metaflex-decoded/data/raw/`, then
`../data/raw` is correct. If not, adjust `RAW` until `Exists: True`.

Do not move on until it is True.

## 1. Inventory — what files do I actually have?

In [ ]:
# List everything under data/raw, whatever the folder structure turns out to be.
all_files = sorted(RAW.rglob("*"))
files = [f for f in all_files if f.is_file()]

print(f"Total files: {len(files)}\n")

# Group by extension
from collections import Counter
ext_counts = Counter(f.suffix.lower() for f in files)
print("By extension:")
for ext, n in ext_counts.most_common():
    print(f"  {ext or '(none)':10s} {n}")

print("\nBy folder:")
folder_counts = Counter(str(f.parent.relative_to(RAW)) for f in files)
for folder, n in sorted(folder_counts.items()):
    print(f"  {folder:30s} {n}")

In [ ]:
# Look at the first 15 filenames so I can see the naming pattern
for f in files[:15]:
    print(f.relative_to(RAW))

In [ ]:
# Read the dataset's own notes first -> added by me
print((RAW / "note.txt").read_text())

In [ ]:
#get real cohort numbers -> added by me
import re

rows = []
for f in all_files:
    m = re.match(r"(\d+)_(\d+)_(\d{8})", f.name)
    if m:
        rows.append({"subject": m.group(1), "visit": int(m.group(2)),
                     "date": pd.to_datetime(m.group(3)),
                     "cohort": f.parent.name, "ext": f.suffix.lower()})

meta = pd.DataFrame(rows)
print("Files parsed:", len(meta), "of", len(all_files))
print("Distinct subjects:", meta["subject"].nunique())
print("\nBy cohort:")
print(meta.groupby("cohort").agg(files=("subject", "size"), subjects=("subject", "nunique")))
print("\nVisits per subject:")
print(meta.groupby("subject").size().value_counts().sort_index())

**TODO — write down in a markdown cell below:**

- How many per-patient CGM files?
- What is the filename pattern? (Remember: patient IDs encode visit number —
  `2001_0` and `2001_1` are the *same person*, two visits.)
- How many `.xls` vs `.xlsx`?

### What I found

- 102 files for patients visited one, 7 files for patients visited twice, 3 files for patients visited 3 times.
- Patiend ID_visit_date
- xls: 101, xlsx: 26

## 2. The summary layer — one row per patient-visit

In [ ]:
# Find the summary files. Adjust the pattern if the names differ.
summary_files = [f for f in files if "summary" in f.name.lower()]
print(summary_files)

In [ ]:
t1 = pd.read_excel(summary_files[0], na_values=["/"])   # check which is which before trusting this
t2 = pd.read_excel(summary_files[1], na_values=["/"])

print("T1 shape:", t1.shape)

print("T2 shape:", t2.shape)
t1.head()

In [ ]:
# Columns side by side — do the two summary sheets agree?
print("In T1 only:", sorted(set(t1.columns) - set(t2.columns)))
print("In T2 only:", sorted(set(t2.columns) - set(t1.columns)))
print("Shared:", len(set(t1.columns) & set(t2.columns)))

In [ ]:
# Missingness per column, as a percentage. This is the single most useful
# table in the whole audit.
def missingness(df, name):
    out = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
    out = out[out > 0]
    print(f"--- {name}: columns with missing values ---")
    print(out.to_string())
    print()
    return out

missingness(t1, "T1DM summary")
missingness(t2, "T2DM summary")

In [ ]:
# The outcome variable. Find the hypoglycemia column and check its balance.
# TODO: replace 'Hypoglycemia' with the actual column name once you have seen it.
hypo_cols = [c for c in t2.columns if "hypo" in str(c).lower()]
print("Candidate columns:", hypo_cols)
#print(t2["Hypoglycemic Agents"]) -> to check the data stored in this col
#print(t2["Hypoglycemia (yes/no)"]) -> to check the data stored in this col

# Once identified:
hypo_cols = "Hypoglycemia (yes/no)"
print(t2[hypo_cols].value_counts(dropna=False))
print(t1[hypo_cols].value_counts(dropna=False))

**TODO — answer in the cell below:**

1. How many summary files, and what are they called?
2. What is the exact name of the hypoglycemia column, and how is it coded?
3. What is the class balance? (This number decides your evaluation metric in Phase 4.)
4. Which clinical columns are so sparse they are unusable?
5. Are units stated anywhere, or assumed? (HbA1c %, glucose mg/dl vs mmol/l — getting this wrong silently ruins every downstream metric.)

**Answers:**
1. 2,  Shanghai_T1DM_Summary.xlsx, Shanghai_T2DM_Summary.xlsx
2. There  are 2: ‘Hypoglycemic Agents' and ’Hypoglycemia (yes/no)’, first with the name of the medicine, second with yes/no
3. Class balance: T1DM: 12.5% no/87.5% yes, T2DM: 90.8% no/9.2% yes, Combine: 80.8% no/19.2% yes
4. The 2-hour postprandial insulin and C-peptide columns are the clear standouts — 37–50% missing across both cohorts That's high enough to call "unusable as a reliable feature" rather than just "somewhat sparse." 
5. Units are documented in header for every column, however the unit of HBA1C is documented in mmol/L instead of % which is more commonly used.

In [ ]:
missingness_series = missingness(t2, "T2DM summary")
threshold = 30  # % missing
sparse_cols = missingness_series[missingness_series > threshold].index.tolist()
print("Columns too sparse to use as features:", sparse_cols)
# check how many POSITIVE cases you'd lose if you dropped rows missing this column
for col in sparse_cols:
    lost = t2[t2[col].isna()][hypo_cols].value_counts()
    print(col, "->", dict(lost))

### What I found

To answer the 4th questions: Which clinical columns are so sparse they are unusable?
I need to fix the missingness funtion -> it didn't catch the cells holds "/"
add: na_values=["/"]

## 3. The CGM layer — one patient file

In [ ]:
# Pick one T1 and one T2 patient file (not a summary) and open them.
cgm_files = [f for f in files if "summary" not in f.name.lower()
             and f.suffix.lower() in (".xls", ".xlsx")]
print(f"{len(cgm_files)} CGM files")
print(cgm_files[0].name)

p = pd.read_excel(cgm_files[0])
print(p.shape)
p.head(20)

In [ ]:
# Column names exactly as they appear — including any Chinese-language duplicates
for c in p.columns:
    print(repr(c))

In [ ]:
# Timestamps: are they regular?
# TODO: replace 'Date' with the real timestamp column name.
ts_col = "Date"

p[ts_col] = pd.to_datetime(p[ts_col], errors="coerce")
gaps = p[ts_col].diff().dt.total_seconds() / 60

print("Recording spans:", p[ts_col].min(), "to", p[ts_col].max())
print("Duration (days):", round((p[ts_col].max() - p[ts_col].min()).total_seconds() / 86400, 2))
print("\nMinutes between readings:")
print(gaps.value_counts().head(10))
print("\nGaps longer than 30 min:", (gaps > 30).sum())

In [ ]:
# The known problem case: patient 2029_0 has missing CGM values from a device swap,
# readings resume 2021-06-03 12:46. Find that file and look at it directly.
target = [f for f in cgm_files if "2029" in f.name]
print(target)

# TODO: open it, locate the gap, and record exactly how long it is.

**TODO — answer:**

- What is the sampling interval, and is it consistent?
- How long is the 2029_0 gap, in hours?
- What fraction of *expected* readings are actually present, for a few patients?
- Are there appended junk rows at the bottom of any file? (Check `.tail(20)`.)

### What I found

*(replace this text)*

## 4. Scale check — loop over all files

In [ ]:
# Do NOT build the cleaning pipeline here. Just collect shapes and basic facts
# so you know what you are dealing with. This may take a minute.

records = []
for f in cgm_files:
    try:
        df = pd.read_excel(f)
        records.append({
            "file": f.name,
            "ext": f.suffix.lower(),
            "rows": len(df),
            "cols": df.shape[1],
            "colnames": tuple(df.columns),
        })
    except Exception as e:
        records.append({"file": f.name, "ext": f.suffix.lower(),
                        "rows": None, "cols": None, "colnames": f"ERROR: {e}"})

inv = pd.DataFrame(records)
print(inv[["rows", "cols"]].describe())
inv.head()

In [ ]:
# Do all files have the same columns? If not, that is a Phase 1 problem to solve.
schema_counts = inv["colnames"].value_counts()
print(f"{len(schema_counts)} distinct column layouts across {len(inv)} files\n")
for schema, n in schema_counts.items():
    print(f"{n} files:")
    print("   ", schema if isinstance(schema, str) else list(schema))
    print()

In [ ]:
# Any files that failed to open?
print(inv[inv["rows"].isna()])

## 5. Cohort summary

**TODO — fill these in. These numbers go straight into the README.**

| Fact | Value |
|---|---|
| Total patients | |
| Total patient-visits | |
| Patients with repeat visits | |
| T1DM / T2DM split | |
| Median recording days per patient | |
| Hypoglycemia positive rate | |
| Distinct column layouts | |
| Files that failed to open | |

## 6. Decisions to carry into Phase 1

Every line here becomes a row in `docs/data-decisions.md`, with a rationale.

**TODO — list them. Starter set, confirm or change each:**

1. Glucose gaps will be **flagged, not imputed** — because imputing glucose invents
   physiology, and it silently changes every variability metric.
2. Resample to a regular 15-minute grid — because Time in Range computed on irregular
   timestamps is a different number, and it has to be reproducible.
3. Drop / rename the duplicate Chinese-language column — decide which, and say why.
4. Repeat visits are **not independent observations** — this constrains cross-validation
   in Phase 4.
5. ...

## What surprised me

*(The most valuable cell in this notebook. Write honestly — the thing you did not
expect is usually the thing worth talking about in an interview.)*